# MediMind AI — QLoRA Fine-tuning on Google Colab

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (Colab Pro) or T4 (free)
2. Add your HuggingFace token in Colab Secrets (key icon on left) as `HF_TOKEN`
3. Run cells top to bottom


In [ ]:
# ── CELL 1: Check GPU ──────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)


In [ ]:
# ── CELL 2: Install dependencies ───────────────────────────────
!pip install -q \
    transformers==4.46.0 \
    peft==0.13.0 \
    trl==0.12.0 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.0 \
    datasets==3.1.0 \
    rouge-score \
    nltk \
    mlflow \
    rich \
    flash-attn --no-build-isolation

print('✓ Dependencies installed')


In [ ]:
# ── CELL 3: HuggingFace login ──────────────────────────────────
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')   # set in Colab Secrets
login(token=HF_TOKEN)
print('✓ Logged in to HuggingFace')


In [ ]:
# ── CELL 4: Clone your repo and download data ─────────────────
!git clone https://github.com/YOUR_USERNAME/medimind-ai.git
%cd medimind-ai

!pip install -q datasets
!python data/scripts/download_all.py
!python data/scripts/preprocess.py
print('✓ Datasets ready')


In [ ]:
# ── CELL 5: Update config with your HF username ──────────────
import yaml

HF_USERNAME = 'your-username'   # ← CHANGE THIS

with open('training/configs/qlora_config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['hub_model_id'] = f'{HF_USERNAME}/medimind-qwen-7b-qlora'
cfg['dataset_path'] = 'data/processed/medimind_train'

with open('training/configs/qlora_config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print(f'✓ Hub model id set to: {cfg["hub_model_id"]}')


In [ ]:
# ── CELL 6: START TRAINING ─────────────────────────────────────
# This will take 4-8 hours on A100 for 3 epochs
# On T4 (free): reduce epochs to 1 in the config first
!python training/train.py --config training/configs/qlora_config.yaml


In [ ]:
# ── CELL 7: Evaluate ──────────────────────────────────────────
!python training/evaluate.py \
    --base_model Qwen/Qwen2.5-7B-Instruct \
    --model_path ./checkpoints/medimind-qwen-7b/final \
    --dataset_path data/processed/medimind_train


In [ ]:
# ── CELL 8: Push to Hub ───────────────────────────────────────
!python training/push_to_hub.py \
    --adapter_path ./checkpoints/medimind-qwen-7b/final \
    --hub_id {HF_USERNAME}/medimind-qwen-7b-qlora
